# Soil Recommendation — Fine-tuning Mistral-7B + QLoRA
### Google Colab — Fresh Training from Scratch
---
| Setting | Value |
|---------|-------|
| Runtime | Google Colab GPU (T4 / A100 / L4) |
| Strategy | Fresh training from base model |
| Dataset | Upload `train.jsonl`, `val.jsonl`, `test.jsonl` to Google Drive |

> **Prerequisites:**
> 1. Set your HuggingFace token in Colab Secrets as `HF_TOKEN`
> 2. Upload your data files to `MyDrive/finetune_data/` on Google Drive
> 3. Select a GPU runtime (Runtime → Change runtime type → GPU)

## STEP 0 — Check Colab GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:800])

## STEP 1 — Install packages (Colab compatible)
> PyTorch already installed on Colab — we only install the training libs

In [ ]:
%%capture
!pip install -q -U pip
!pip install -q -U bitsandbytes --extra-index-url https://pypi.nvidia.com
!pip install -q -U triton
!pip install -q \
    numpy==1.26.4 \
    transformers==4.44.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    accelerate==0.33.0 \
    datasets==2.20.0 \
    sentencepiece==0.2.0 \
    rouge-score \
    rich
print('Done - Libraries installed')

## STEP 2 — Verify GPU & libraries

In [ ]:
import torch
import numpy as np
import transformers, bitsandbytes, peft, trl, datasets

print(f'numpy          : {np.__version__}')
print(f'torch          : {torch.__version__}')
print(f'transformers   : {transformers.__version__}')
print(f'bitsandbytes   : {bitsandbytes.__version__}')
print(f'peft           : {peft.__version__}')
print(f'trl            : {trl.__version__}')
print(f'datasets       : {datasets.__version__}')
print()

assert torch.cuda.is_available(), 'No GPU! Change runtime to GPU.'
print(f'GPU            : {torch.cuda.get_device_name(0)}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA version   : {torch.version.cuda}')
print('\nAll OK')

## STEP 3 — Mount Google Drive & copy data locally

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive', force_remount=False)

# ══════════════════════════════════════════
# Edit these paths if your Drive layout differs
# ══════════════════════════════════════════
DATA_DRIVE_PATH = '/content/drive/MyDrive/finetune_data'
OUTPUT_DIR      = '/content/drive/MyDrive/mistral_soil_output'

# Copy data to local /content for faster I/O during training
LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)
for fname in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    src = os.path.join(DATA_DRIVE_PATH, fname)
    dst = os.path.join(LOCAL_DATA, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f'  Copied: {fname}  ({size:.1f} MB)')
    else:
        print(f'  WARNING: {fname} not found at {src}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'\nOutput directory: {OUTPUT_DIR}')
print('All paths OK')

## STEP 4 — Hugging Face login

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('Logged in via Colab Secrets')
except Exception as e:
    print(f'Colab Secrets not available ({e}), falling back to manual login')
    login(add_to_git_credential=False)
    print('Logged in manually')

## STEP 5 — Configuration

In [ ]:
import os, torch
from dataclasses import dataclass, field
from typing import List

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9

@dataclass
class Config:
    # Base model — Mistral 7B Instruct
    model_name : str = 'mistralai/Mistral-7B-Instruct-v0.2'
    output_dir : str = OUTPUT_DIR

    # QLoRA settings
    lora_r          : int   = 16
    lora_alpha      : int   = 32
    lora_dropout    : float = 0.05
    target_modules  : List[str] = field(default_factory=lambda: [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ])

    # Training — auto-adjusted for available VRAM
    num_epochs      : int   = 3
    batch_size      : int   = 4 if VRAM_GB >= 40 else 2   # A100: 4, T4: 2
    grad_accum      : int   = 4 if VRAM_GB >= 40 else 8   # effective batch = 16
    learning_rate   : float = 2e-4
    lr_scheduler    : str   = 'cosine'
    warmup_ratio    : float = 0.05
    max_seq_length  : int   = 512
    weight_decay    : float = 0.01
    grad_clip       : float = 1.0

    # Logging & checkpointing
    logging_steps    : int = 10
    eval_steps       : int = 200
    save_steps       : int = 200
    save_total_limit : int = 3

    train_file : str = '/content/data/train.jsonl'
    val_file   : str = '/content/data/val.jsonl'
    seed       : int = 42

cfg = Config()

print(f'VRAM           : {VRAM_GB:.1f} GB')
print(f'Batch size     : {cfg.batch_size}')
print(f'Grad accum     : {cfg.grad_accum}')
print(f'Effective batch: {cfg.batch_size * cfg.grad_accum}')
print(f'Max seq length : {cfg.max_seq_length}')
print(f'Learning rate  : {cfg.learning_rate}')
print(f'Epochs         : {cfg.num_epochs}')
print(f'Output dir     : {cfg.output_dir}')
print('Config OK')

## STEP 6 — Format dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files={
    'train'     : cfg.train_file,
    'validation': cfg.val_file,
})

print(f'Train      : {len(raw["train"]):,}')
print(f'Validation : {len(raw["validation"]):,}')

# Character replacements to remove emojis/special chars that confuse the tokenizer
REPLACEMENTS = {
    '━': '-', '─': '-', '═': '=',
    '📋': '', '📊': '', '🧪': '',
    '✅': '[OK]', '🔴': '[LOW]', '🟡': '[HIGH]',
    '⚠': '[!]', '💧': '', '🇪🇬': '',
    '\u200f': '', '\u200e': '',
}

def format_prompt(example):
    instruction = example['instruction'].strip()
    inp         = example.get('input', '').strip()
    output      = example['output'].strip()

    for old, new in REPLACEMENTS.items():
        instruction = instruction.replace(old, new)
        output      = output.replace(old, new)
        if inp:
            inp = inp.replace(old, new)

    if inp:
        text = f'[INST] {instruction}\n\n{inp} [/INST]\n{output}</s>'
    else:
        text = f'[INST] {instruction} [/INST]\n{output}</s>'

    return {'text': text}

dataset = raw.map(
    format_prompt,
    remove_columns=raw['train'].column_names,
    num_proc=2,
)

print(f'\nSample (300 chars):')
print(dataset['train'][0]['text'][:300])
print('Dataset OK')

## STEP 7 — Load base model in 4-bit (QLoRA)

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name,
    trust_remote_code=True,
    use_fast=False,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer OK')

# Base model in 4-bit
print('Loading base model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model.config.use_cache      = False
model.config.pretraining_tp = 1
print('Base model OK')

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nVRAM used : {used:.1f} / {total:.1f} GB ({100*used/total:.0f}%)')

## STEP 8 — Apply QLoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare model for k-bit training (freezes base weights, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=cfg.target_modules,
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable/1e6:.1f}M  ({100*trainable/total_p:.2f}%)')
print(f'Total params     : {total_p/1e9:.2f}B')
print('QLoRA applied OK')

## STEP 9 — Training arguments

In [ ]:
from transformers import TrainingArguments

# A100/H100 support bf16; T4/V100 use fp16
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8

training_args = TrainingArguments(
    output_dir=cfg.output_dir,

    # Loop
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,

    # Optimizer
    optim='paged_adamw_32bit',
    learning_rate=cfg.learning_rate,
    lr_scheduler_type=cfg.lr_scheduler,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.grad_clip,

    # Precision
    fp16=not use_bf16,
    bf16=use_bf16,

    # Logging & checkpointing
    logging_steps=cfg.logging_steps,
    evaluation_strategy='steps',
    eval_steps=cfg.eval_steps,
    save_strategy='steps',
    save_steps=cfg.save_steps,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Memory optimizations
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_num_workers=2,
    group_by_length=True,

    # Misc
    seed=cfg.seed,
    report_to='none',
    ddp_find_unused_parameters=False,
)

steps_per_epoch = len(dataset['train']) // (cfg.batch_size * cfg.grad_accum)
print(f'Precision      : {"bf16" if use_bf16 else "fp16"}')
print(f'Steps/epoch    : {steps_per_epoch}')
print(f'Total steps    : {steps_per_epoch * cfg.num_epochs}')
print(f'Save every     : {cfg.save_steps} steps → Drive')
print('Training args OK')

## STEP 10 — Start training
> Checkpoints are saved to Google Drive every `save_steps` steps so you never lose progress if the session disconnects.

In [ ]:
from trl import SFTTrainer
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    dataset_text_field='text',
    max_seq_length=cfg.max_seq_length,
    packing=False,
)

torch.cuda.empty_cache()

print('Training started...')
print(f'Output: {cfg.output_dir}')
print('-' * 50)

start  = time.time()
result = trainer.train()
elapsed = (time.time() - start) / 3600

print(f'\nDone in {elapsed:.2f} hours')
print(f'Final train loss: {result.training_loss:.4f}')

## STEP 11 — Save final model to Drive
> Saves the best model weights so you can load them for inference without re-training.

In [ ]:
import os

final_dir = os.path.join(cfg.output_dir, 'final')
os.makedirs(final_dir, exist_ok=True)

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print(f'Model saved to: {final_dir}')
print('\nFiles:')
for f in sorted(os.listdir(final_dir)):
    mb = os.path.getsize(os.path.join(final_dir, f)) / 1e6
    print(f'  {f}: {mb:.1f} MB')

## STEP 12 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt, os

logs        = trainer.state.log_history
train_steps = [x['step'] for x in logs if 'loss'      in x and 'eval_loss' not in x]
train_loss  = [x['loss'] for x in logs if 'loss'      in x and 'eval_loss' not in x]
eval_steps  = [x['step'] for x in logs if 'eval_loss' in x]
eval_loss   = [x['eval_loss'] for x in logs if 'eval_loss' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

ax1.plot(train_steps, train_loss, color='#1D9E75', linewidth=1.5)
ax1.set_title('Training Loss', fontweight='bold')
ax1.set_xlabel('Steps'); ax1.set_ylabel('Loss'); ax1.grid(alpha=0.3)

ax2.plot(eval_steps, eval_loss, color='#2E6DA4', linewidth=2, marker='o', markersize=4)
ax2.set_title('Validation Loss', fontweight='bold')
ax2.set_xlabel('Steps'); ax2.set_ylabel('Loss'); ax2.grid(alpha=0.3)

if eval_loss:
    best_idx = eval_loss.index(min(eval_loss))
    ax2.axvline(eval_steps[best_idx], color='red', linestyle='--',
                label=f'Best: {min(eval_loss):.4f}')
    ax2.legend()

plt.suptitle('Mistral-7B QLoRA — Soil Fertilization Recommendation', fontweight='bold')
plt.tight_layout()
out_png = os.path.join(cfg.output_dir, 'training_curves.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')

## STEP 13 — Inference test

In [ ]:
model.eval()
torch.cuda.empty_cache()

def ask(instruction, max_new_tokens=512):
    prompt = f'[INST] {instruction} [/INST]'
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=400
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

q1 = ('لدي تربة زراعية اريد زراعة قمح فيها في الموسم الشتوي.\n'
      'التحليل: N=100 kg/ha, P=25 kg/ha, K=0 kg/ha, pH=7.8\n'
      'نوع التربة: طينية لومية. اعداد ورقة توصية.')
print('TEST 1 — Wheat (Winter)')
print('-' * 50)
print(ask(q1))

print('\n' + '=' * 50)

q2 = 'تحليل تربة قصب السكر صيفي: N=180 P=50 K=80 pH=7.2. ما التوصية؟'
print('TEST 2 — Sugarcane (Summer)')
print('-' * 50)
print(ask(q2))

## STEP 14 — ROUGE evaluation

In [ ]:
from rouge_score import rouge_scorer as rs
import json, random
import numpy as np

test_data = []
with open('/content/data/test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f'Test set size: {len(test_data)}')

random.seed(42)
sample = random.sample(test_data, min(50, len(test_data)))
scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

r1, r2, rL = [], [], []
for i, item in enumerate(sample):
    if i % 10 == 0:
        print(f'  Evaluating {i}/{len(sample)}...')
    pred   = ask(item['instruction'], max_new_tokens=700)
    scores = scorer.score(item['output'], pred)
    r1.append(scores['rouge1'].fmeasure)
    r2.append(scores['rouge2'].fmeasure)
    rL.append(scores['rougeL'].fmeasure)

print(f'\nROUGE-1 : {np.mean(r1):.4f}')
print(f'ROUGE-2 : {np.mean(r2):.4f}')
print(f'ROUGE-L : {np.mean(rL):.4f}')